# Finding an attacking left-back in the Indian Super League (2021/22)

A data-driven scouting exercise using StatsBomb open data.

**Question.** A modest club can't buy the €40M full-back. It goes to undervalued
markets to find 80% of the output for 10% of the fee. Within one fully-covered
league, who is the most productive *attacking* left-back?

**Why this league.** Coverage due-diligence (`scripts/coverage_check.py`) ruled
out the alternatives: MLS 2023 open data is only 6 matches, and La Liga open
data is Barcelona's matches only. The Indian Super League 2021/22 is the one
option with a full season (115 matches, 11 teams) — a rankable pool.

**Method.**
1. Load and cache the season's events.
2. Build a clean pool of *genuine* left-backs (modal position = LB/LWB, ≥ 900').
3. Compute attacking metrics per 90 minutes.
4. Rank within the league as percentiles.

**Caveats stated up front.** The pool is small (N ≈ 13), so percentiles are
coarse — read them as "2nd of 13", not "top 5%". Metrics are per-90 within one
league; cross-league comparison would need a league-strength adjustment that
open data doesn't provide. Two players are positional hybrids (low `share_LB`)
and are flagged, not hidden.

## 1. Setup and load

In [ ]:
import sys
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# resolve repo root wherever the notebook runs from (it lives in notebooks/)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))          # so we can import the src/ package

from src.isl import load_isl_events, build_left_back_pool

ev = load_isl_events(ROOT / "data/raw/isl_2122_events.pkl")
print("Events:", ev.shape)

## 2. Clean pool of left-backs

`build_left_back_pool` keeps only players whose *modal* position is LB/LWB and
who cleared 900 minutes. `share_LB` shows how much of each player's time was
actually spent at left-back — the two low values are hybrids to flag later.

In [ ]:
pool = build_left_back_pool(ev, min_minutes=900)
print(f"Pool: {len(pool)} left-backs with >=900 min")
pool

## 3. Unpack location lists into flat columns

StatsBomb stores position as a `[x, y]` list inside a single cell, and many
events have no location (`NaN`). `coord` returns the i-th element for lists and
`NaN` otherwise, so we get clean numeric `x`, `y`, `end_x`... columns to filter
on. The pitch is **120 x 80**.

In [ ]:
def coord(v, i):
    return v[i] if isinstance(v, (list, tuple)) else np.nan

ev["x"]           = ev["location"].apply(lambda v: coord(v, 0))
ev["y"]           = ev["location"].apply(lambda v: coord(v, 1))
ev["end_x"]       = ev["pass_end_location"].apply(lambda v: coord(v, 0))
ev["end_y"]       = ev["pass_end_location"].apply(lambda v: coord(v, 1))
ev["carry_end_x"] = ev["carry_end_location"].apply(lambda v: coord(v, 0))

## 4. Attacking metrics, normalised per 90

Metrics chosen because event data captures them with little ambiguity for a
full-back: crosses (and completion %), open-play passes into the box,
shot assists (a pass leading to a shot — our free proxy for xA), progressive
carries, and final-third touches.

Two StatsBomb conventions to remember: a **completed** pass has an *empty*
`pass_outcome` (so `.isna()` means completed), and an **open-play** pass has an
empty `pass_type`. The box is `x >= 102`, `y` in 18–62; the final third is
`x >= 80`.

In [ ]:
evp = ev[ev["player"].isin(pool.index)].copy()

open_play = evp["pass_type"].isna()      # not corner / free-kick / throw-in
completed = evp["pass_outcome"].isna()   # empty outcome = completed pass

evp["cross"]       = (evp["pass_cross"] == True) & open_play
evp["cross_cmp"]   = evp["cross"] & completed
evp["box_pass"]    = ((evp["type"] == "Pass") & open_play & completed
                      & (evp["end_x"] >= 102) & evp["end_y"].between(18, 62)
                      & (evp["x"] < 102))
evp["shot_assist"] = (evp["pass_shot_assist"] == True)
evp["prog_carry"]  = (evp["type"] == "Carry") & ((evp["carry_end_x"] - evp["x"]) >= 5)
evp["f3_touch"]    = (evp["x"] >= 80)

flags = ["cross", "cross_cmp", "box_pass", "shot_assist", "prog_carry", "f3_touch"]
totals = evp.groupby("player")[flags].sum()

df = pool.join(totals)
per90 = df[flags].div(df["minutes"], axis=0).mul(90).round(2)
per90.columns = ["crosses", "crosses_cmp", "box_passes", "shot_assists",
                 "prog_carries", "f3_touches"]
per90["cross_pct"] = (df["cross_cmp"] / df["cross"] * 100).round(0)

profile = pool[["team", "minutes", "share_LB"]].join(per90)
profile.sort_values("box_passes", ascending=False)

## 5. Percentiles within the league

`rank(pct=True)` turns each metric into its percentile among the 13. With this
sample size, phrase findings as ranks ("2nd of 13"), and asterisk the hybrids
(low `share_LB`) whose per-90 blends minutes from another position.

In [ ]:
metrics = ["crosses", "box_passes", "shot_assists", "prog_carries", "f3_touches"]
pct = profile[metrics].rank(pct=True).mul(100).round(0)
pct.columns = [c + "_pctl" for c in metrics]
pct.join(profile[["team", "minutes", "share_LB"]]).sort_values("box_passes_pctl", ascending=False)

## 6. Next — visualise the standout

Once the target is chosen, build a radar / pizza chart with `mplsoccer` and a
written positional evaluation. That figure and write-up become the featured
piece linked from the README.

In [ ]:
# mplsoccer visualisation goes here